In [ ]:
# Import libraries we need
import pandas as pd
import numpy as np
import os
from pathlib import Path


def find_data_file(filename: str) -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        matches = list(base.rglob(filename))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"Could not find {filename} from {start}")


# Show current location
print("Current folder:", os.getcwd())

# Load the dataset
csv_path = find_data_file("Mbeya_soil_dataset_final.csv")
print("Using data file:", csv_path)
df = pd.read_csv(csv_path)

# Clean column names - remove newline characters
df.columns = df.columns.str.replace("\n", " ").str.strip()

# Rename target column
df = df.rename(columns={"Recommendation ? TARGET": "Recommendation"})

# Show basic info
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nMissing values:", df.isnull().sum().sum())
print("\nTarget distribution:")
print(df["Recommendation"].value_counts())

Current folder: c:\Users\XLYGER\Desktop\fertilizer_dss\ml
Shape: (4998, 13)

Columns: ['District', 'Latitude', 'Longitude', 'pH', 'N_kg_ha', 'P_kg_ha', 'K_kg_ha', 'Soil_Type', 'Temperature_C', 'Humidity_pct', 'Rainfall_mm', 'Recommendation', 'When_to_Apply']

Missing values: 0

Target distribution:
Recommendation
No Fertilizer    2324
Urea             1213
NPK               947
Apply Lime        352
DAP               162
Name: count, dtype: int64


In [ ]:
# Show first 5 rows of data
print("First 5 rows:")
df.head()

First 5 rows:


,District,Season,Soil Type,N (mg/kg),P (mg/kg),K (mg/kg),Temperature (°C),Humidity (%),pH,Rainfall (mm),Recommendation
0,Mbeya Rural,Short Rains,Nitisol,42,17,65,23,72,4.9,425,Apply Lime
1,Busokelo,Long Rains,Nitisol,41,17,35,17,65,5.4,425,Apply Lime
2,Busokelo,Long Rains,Nitisol,50,28,30,17,83,5.1,425,Apply Lime
3,Busokelo,Long Rains,Alluvial Clay,73,29,64,22,76,6.8,236,NPK
4,Busokelo,Short Rains,Andosol,55,24,51,16,77,6.3,425,DAP


In [ ]:
# Create two new features that help the model
# NPK_Total = total nutrients in soil
df["NPK_Total"] = df["N (mg/kg)"] + df["P (mg/kg)"] + df["K (mg/kg)"]

# NP_Ratio = ratio of Nitrogen to Phosphorus
df["NP_Ratio"] = df["N (mg/kg)"] / (df["P (mg/kg)"] + 1)

# Show result
print("New columns added!")
print(df[["N (mg/kg)", "P (mg/kg)", "K (mg/kg)", "NPK_Total", "NP_Ratio"]].head())

New columns added!
   N (mg/kg)  P (mg/kg)  K (mg/kg)  NPK_Total  NP_Ratio
0         42         17         65        124  2.333333
1         41         17         35         93  2.277778
2         50         28         30        108  1.724138
3         73         29         64        166  2.433333
4         55         24         51        130  2.200000


In [ ]:
# Import LabelEncoder
from sklearn.preprocessing import LabelEncoder

# Columns that have words (not numbers)
cat_columns = ["District", "Season", "Soil Type"]

# Create encoder object
le = LabelEncoder()

# Convert each text column to numbers
for col in cat_columns:
    df[col] = le.fit_transform(df[col])

# Show result
print("Text converted to numbers!")
print(df[["District", "Season", "Soil Type"]].head())

Text converted to numbers!
   District  Season  Soil Type
0         4       1          4
1         0       0          4
2         0       0          4
3         0       0          0
4         0       1          1


In [ ]:
# Encode target column separately
# We save this encoder to use later when showing results
le_target = LabelEncoder()

# Convert Recommendation to numbers
df["Recommendation"] = le_target.fit_transform(df["Recommendation"])

# Show what each number means
print("Classes and their numbers:")
for i, cls in enumerate(le_target.classes_):
    print(f"  {i} = {cls}")

print("\nTarget column sample:")
print(df["Recommendation"].head())

Classes and their numbers:
  0 = Apply Lime
  1 = DAP
  2 = NPK
  3 = No Fertilizer
  4 = Urea

Target column sample:
0    0
1    0
2    0
3    2
4    1
Name: Recommendation, dtype: int64


In [ ]:
# Import train_test_split
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop("Recommendation", axis=1)
y = df["Recommendation"]

# Split data - 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Show sizes
print("Training size:", X_train.shape)
print("Testing size:", X_test.shape)

Training size: (3558, 12)
Testing size: (890, 12)


In [ ]:
# Import Random Forest
from sklearn.ensemble import RandomForestClassifier

# Create the model
model = RandomForestClassifier(
    n_estimators=100,  # number of trees
    random_state=42    # same results every time
)

# Train the model
model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [ ]:
# Import metrics
from sklearn.metrics import accuracy_score, classification_report

# Make predictions on test data
y_pred = model.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy * 100:.2f}%")

# Show detailed report
print("\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

Model Accuracy: 95.62%

Detailed Report:
               precision    recall  f1-score   support

   Apply Lime       0.98      0.99      0.99       173
          DAP       0.95      0.92      0.93       180
          NPK       0.96      0.94      0.95       180
No Fertilizer       0.97      0.99      0.98       177
         Urea       0.92      0.94      0.93       180

     accuracy                           0.96       890
    macro avg       0.96      0.96      0.96       890
 weighted avg       0.96      0.96      0.96       890



In [ ]:
# Import joblib to save model
import joblib
import os

# Create folder to save model
os.makedirs("model", exist_ok=True)

# Save the model
joblib.dump(model, "model/fertilizer_model.pkl")

# Save the label encoder for target
joblib.dump(le_target, "model/label_encoder.pkl")

# Save feature names
feature_names = X.columns.tolist()
joblib.dump(feature_names, "model/feature_names.pkl")

print("Model saved successfully!")
print("Files saved:")
print("  - model/fertilizer_model.pkl")
print("  - model/label_encoder.pkl")
print("  - model/feature_names.pkl")

Model saved successfully!
Files saved:
  - model/fertilizer_model.pkl
  - model/label_encoder.pkl
  - model/feature_names.pkl
